In [18]:
import os 

os.getcwd()

'/home/leostre/Рабочий стол/py-boost/rebuttal'

In [17]:
os.chdir('../rebuttal/')

In [3]:
os.chdir('../5.2/')

In [19]:
COMPUTATIONAL_METRICS = ['duration_seconds', 'inference_time', 'mean_leaves', 'mean_nodes', 'ntrees',
        'train_time', 'get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']
DETAILED_COMPUTATIONAL = ['get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']

In [20]:
import mlflow 

from mlflow.client import MlflowClient 

client = MlflowClient()

In [21]:
client.search_experiments()

[<Experiment: artifact_location='file:///app/mlruns/301780227554818042', creation_time=1774469388438, experiment_id='301780227554818042', last_update_time=1774469388438, lifecycle_stage='active', name='baselines_5.2.1', tags={}>,
 <Experiment: artifact_location='file:///app/mlruns/756291063253549330', creation_time=1774469376945, experiment_id='756291063253549330', last_update_time=1774469376945, lifecycle_stage='active', name='hyperbolic_5.2.1', tags={}>,
 <Experiment: artifact_location='file:///app/mlruns/997958599129062246', creation_time=1774469370672, experiment_id='997958599129062246', last_update_time=1774469370672, lifecycle_stage='active', name='sigmoid_5.2.1', tags={}>,
 <Experiment: artifact_location='file:///app/mlruns/0', creation_time=1774469370663, experiment_id='0', last_update_time=1774469370663, lifecycle_stage='active', name='Default', tags={}>]

In [22]:
def filter_df(df, rule):
    for k, v in rule.items():
        if k not in df.columns:
            continue
        df = df[df[k] == v]
    return df

In [23]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_all_runs_data(experiment_names=None, include_artifacts=False):
    """
    Extract all runs data from MLflow experiments into a comprehensive DataFrame
    
    Args:
        experiment_names: List of experiment names or None for all experiments
        include_artifacts: Whether to include artifact URIs
    
    Returns:
        DataFrame with all runs data
    """
    client = MlflowClient()
    
    # Get experiments
    if experiment_names is None:
        experiments = client.search_experiments()
    else:
        experiments = [client.get_experiment_by_name(name) for name in experiment_names]
        experiments = [exp for exp in experiments if exp is not None]
    
    all_runs_data = []
    
    for experiment in tqdm(experiments, desc="Processing experiments"):
        experiment_id = experiment.experiment_id
        experiment_name = experiment.name
        
        print(f"Processing experiment: {experiment_name}")
        
        # Get all runs for this experiment
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            max_results=10000  # Adjust if you have more runs
        )
        
        for run in tqdm(runs, desc=f"Runs in {experiment_name}", leave=False):
            run_data = {
                'run_id': run.info.run_id,
                'experiment_id': experiment_id,
                'experiment_name': experiment_name,
                'run_name': run.data.tags.get('mlflow.runName', ''),
                'status': run.info.status,
                'start_time': pd.to_datetime(run.info.start_time, unit='ms'),
                'end_time': pd.to_datetime(run.info.end_time, unit='ms') if run.info.end_time else None,
                'duration_seconds': (run.info.end_time - run.info.start_time) / 1000.0 if run.info.end_time and run.info.start_time else None,
            }
            
            # Add parameters
            for key, value in run.data.params.items():
                run_data[f'param_{key}'] = value
            
            # Add metrics
            for key, value in run.data.metrics.items():
                run_data[f'metric_{key}'] = value
            
            # Add tags
            for key, value in run.data.tags.items():
                if key not in ['mlflow.runName', 'mlflow.user']:
                    run_data[f'tag_{key}'] = value
            
            # Add artifact location if requested
            if include_artifacts:
                run_data['artifact_uri'] = run.info.artifact_uri
            
            all_runs_data.append(run_data)
    
    return pd.DataFrame(all_runs_data)

def get_baselines(path, version):
    start_dir = os.getcwd()
    os.chdir(path)
    df = get_all_runs_data(['baselines_' + str(version)])
    os.chdir(start_dir)
    return df 





In [24]:
# bsln = get_baselines('/home/leostre/Рабочий стол/py-boost/5.2', '5.2')

In [25]:
EXCLUDE = {'experiment_id', 'experiment_name', 'status', 'start_time', 'end_time', 'run_name', 'tag_mlflow.source.name', 'tag_mlflow.source.git.commit',
       'tag_mlflow.source.type', 'param_error', 'tag_status',
       'param_total_runs', 'param_successful_runs', 'mean_f1', 'mean_accuracy', 'param_n_splits', 'param_n_successful_folds'}

def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    print(after_exclusion_by_name)
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col or col in ('param_stabilization_threshold', 'param_smoothing_alpha')
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    print(metric_cols)
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    print(id_cols)
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df


In [94]:
def flatten_index(df):
    columns = df.columns
    if isinstance(columns, pd.MultiIndex):
        columns = [c[0] if not c[1] else c[1] for c in columns] 
        columns = [c if c != 'mean' else 'value' for c in columns]
    df.columns = columns  

In [26]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
def comprehensive_statistical_test(df1: pd.DataFrame, df2: pd.DataFrame, value_cols=None):
    """
    Comprehensive statistical testing between two DataFrames
    
    Parameters:
    - value_cols: List of value columns to test (default: ['value', 'duration_seconds'])
    """
    if value_cols is None:
        value_cols = ['value', 'duration_seconds']
    
    group_cols = ['dataset', 'lr', 'subsample', 'sketch_method', 'sketch_outputs', 'metric']
    
    # Merge DataFrames
    merged = pd.merge(
        df1, 
        df2, 
        on=group_cols,
        suffixes=('_df1', '_df2'),
        how='inner'
    )
    
    all_results = {}
    
    for value_col in value_cols:
        if f"{value_col}_df1" not in merged.columns or f"{value_col}_df2" not in merged.columns:
            continue
            
        results = []
        
        for metric in merged['metric'].unique():
            metric_data = merged[merged['metric'] == metric]
            
            if len(metric_data) < 2:
                continue
                
            # Extract values
            values_df1 = metric_data[f"{value_col}_df1"].values
            values_df2 = metric_data[f"{value_col}_df2"].values
            
            # Paired t-test
            t_stat, p_value = stats.ttest_rel(values_df1, values_df2)
            
            # Calculate statistics
            differences = values_df1 - values_df2
            mean_diff = np.mean(differences)
            std_diff = np.std(differences, ddof=1)
            cohens_d = mean_diff / std_diff if std_diff > 0 else 0
            
            n = len(differences)
            se_diff = std_diff / np.sqrt(n)
            ci_low = mean_diff - 1.96 * se_diff
            ci_high = mean_diff + 1.96 * se_diff
            
            results.append({
                'metric': metric,
                'value_column': value_col,
                'n_pairs': n,
                'mean_df1': np.mean(values_df1),
                'mean_df2': np.mean(values_df2),
                'mean_difference': mean_diff,
                'std_difference': std_diff,
                't_statistic': t_stat,
                'p_value': p_value,
                'cohens_d': cohens_d,
                'ci_low': ci_low,
                'ci_high': ci_high,
                'significant_0.05': p_value < 0.05
            })
        
        results_df = pd.DataFrame(results)
        
        # Adjust p-values
        if not results_df.empty:
            rejected, pvals_corrected, _, _ = multipletests(
                results_df['p_value'].values, 
                alpha=0.05, 
                method='fdr_bh'
            )
            results_df['p_value_adj'] = pvals_corrected
            results_df['significant_adj'] = rejected
        
        all_results[value_col] = results_df
    
    return all_results, merged

In [27]:
# def statistical_test()

import scipy.stats as stats

test_fn = lambda x, y: stats.ttest_rel(x, y).pvalue

test_fn = lambda x, y: stats.ttest_ind_from_stats(
    np.mean(x), np.std(x), len(x), np.mean(y), np.std(y), len(y)
).pvalue

def ttest(df1, df2, p=0.05):
    gr_cols = ['dataset', 'lr', 'subsample',
       'sketch_method', 'sketch_outputs', 'metric',]
    to_cat = []
    for df in [df1, df2]:
        gr_df = df.groupby(gr_cols)['value'].aggregate(list)
        to_cat.append(gr_df.map(np.array))
    result = pd.Series([test_fn(x1, x2) for x1, x2 in zip(*to_cat)], index=gr_df.index) < p
    return result



## Compare experiments

In [28]:
# DATASET = [
#     # 'mediamill',
#     'mnist',
#     'cifar10',
#     # 'yeast', 
#         #    'age_prediction'
#         #    'birds', 
#         #    'genbase'
#     # 'mbd',
# ]


# dfs = [get_all_runs_data([exp]) for exp in EXPS]

# processed = {}
# for exp, df in zip(EXPS, dfs):
#     df = df[df.param_dataset.isin(DATASET)]
#     fp_df = filter_data(df)
#     mlt_df = melt_metrics(fp_df)
#     agg_mtrs = mlt_df.groupby(['dataset', 'sketch_method', 'sketch_outputs', 'subsample', 'lr',
#                                'stabilization_threshold',
#                                 'smoothing_alpha',
#                                  'metric']).agg({'value': 'mean'})#.reset_index()
#     processed[exp] = (agg_mtrs)




In [93]:
DATASET = [
    # 'mediamill',
    # 'mnist',
    # 'cifar10',
    # 'yeast', 
        #    'age_prediction'
        #    'birds', 
        #    'genbase'
    # 'mbd',
]

EXPS = [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    # 'sigmoid_5.2',
]
TO_INT = ['sketch_outputs']
TO_FLOAT = ['smoothing_alpha', 'stabilization_threshold', 'subsample', 'lr', ]

dfs = [get_all_runs_data([exp]) for exp in EXPS] 

version = '5.2'
if version:
    dfs += [get_baselines(f'../{version}', version)]
    EXPS += [f'baselines_{version}']

processed = {}
for exp, df in zip(EXPS, dfs):
    if DATASET:
        df = df[df.param_dataset.isin(DATASET)]
    df = filter_data(df)
    mlt_df = melt_metrics(df)
    agg_mtrs = mlt_df.groupby(['dataset',
                                'sketch_method', 'sketch_outputs', 
                                *(['smoothing_alpha', 'stabilization_threshold'] if 'smoothing_alpha' in mlt_df.columns else []),
                                'subsample', 'lr', 'metric', ]).agg({'value': ['mean', 'std']})#.reset_index()
    flatten_index(agg_mtrs)
    agg_mtrs['experiment'] = exp
    processed[exp] = (agg_mtrs)

Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: sigmoid_5.2.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: hyperbolic_5.2.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: baselines_5.2.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: baselines_5.2


Processing experiments: 100%|██████████| 1/1 [00:00<00:00, 13.43it/s]


['run_id', 'duration_seconds', 'param_es', 'param_max_bin', 'param_dataset', 'param_sketch_params', 'param_ntrees', 'param_lr', 'param_colsample', 'param_min_data_in_bin', 'param_subsample', 'param_min_gain_to_split', 'param_stabilization_threshold', 'param_sketch_method', 'param_callbacks', 'param_gd_steps', 'param_lambda_l2', 'param_quantization', 'param_use_hess', 'param_verbose', 'param_sketch_outputs', 'param_loss', 'param_smoothing_alpha', 'param_min_data_in_leaf', 'param_quant_sample', 'param_max_depth', 'param_metric', 'param_seed', 'metric_recall_fold_0', 'metric_exact_match_fold_0', 'metric_threshold_value_min_fold_0', 'metric_ntrees_fold_0', 'metric_precision_fold_0', 'metric_f1_micro_fold_0', 'metric_inference_time_fold_0', 'metric_threshold_value_mean_fold_0', 'metric_train_time_fold_0', 'metric_f1_macro_fold_0', 'metric_roc_auc_fold_0', 'metric_threshold_value_max_fold_0', 'metric_mean_nodes_fold_0', 'metric_adaptive_threshold_fallback_fold_0', 'metric_f1_fold_0', 'metric

In [30]:
import matplotlib.pyplot as plt 
import seaborn as sns 


def metric_heatmaps(df, x, y):
    # Pivot the data to create a matrix for each metric
    pivot_dfs = {}
    metrics = df['metric'].unique()

    for metric in metrics:
        # Filter for the specific metric
        metric_df = df[df['metric'] == metric].copy()
        
        # Pivot to create matrix
        pivot = metric_df.pivot_table(
            index=x, 
            columns=y, 
            values='value',
            aggfunc='mean'
        )
        pivot_dfs[metric] = pivot

    # Create heatmaps for each metric
    fig, axes = plt.subplots((len(metrics) + 3) // 4, 4, figsize=(20, 10))
    axes = axes.flatten()

    for i, (metric, pivot_df) in enumerate(pivot_dfs.items()):
        if i < len(axes):
            sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[i],
                    cbar_kws={'label': metric})
            axes[i].set_title(f'{metric}')
            axes[i].set_xlabel(y)
            axes[i].set_ylabel(x)

    # Hide any unused subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()




In [31]:
metric_heatmaps(
    processed['sigmoid_5.2'].reset_index().groupby(['smoothing_alpha', 'stabilization_threshold', 'metric'])['value'].aggregate('mean').reset_index(), 'smoothing_alpha', 'stabilization_threshold'
)

KeyError: 'sigmoid_5.2'

## setting up the other HPs

sthreshold for correct comparison

to create baselines for corruption experiments

In [21]:
SLICE = dict(
    smoothing_alpha=0.9,
    stabilization_threshold=1.0,
)

for exp in processed:
    df = processed[exp]
    fdf = filter_df(df.reset_index(), SLICE)
    # .drop(SLICE.keys(), axis=1)
    if 'index' in fdf.columns:
        fdf.drop('index', axis=1, inplace=True)
    # processed[exp] = fdf.set_index([c for c in fdf.columns if c not in ('value',)])

In [222]:
# for exp in processed:
#     processed[exp].to_csv(f'no_corruption_{exp}.csv')

In [62]:
import numpy as np 


def compare_prepare(modified, base, filter, method='standard', include_detailed=False):
    modified_data = processed[modified].reset_index()
    if base:
        basic = processed[base].reset_index()
    else:
        basic = modified_data.copy()
        basic.loc[:, :] = np.zeros(modified_data.shape)
    flatten_index(basic)
    flatten_index(modified_data)
    basic = basic.drop('std', axis=1)
    modified_data = modified_data.drop('std', axis=1)
    hps = set(modified_data.columns) & set(basic.columns)
    hps.discard('value')
    hps = list(hps)
    difference = pd.merge(modified_data, basic, on=hps, suffixes=['_m', '_b'])
    metrics = difference['metric']
    difference['value'] = (difference['value_m'] - difference['value_b']) * 100
    
    if include_detailed:
        is_computational = metrics.isin(COMPUTATIONAL_METRICS).values
    else:
        is_computational = (metrics.isin(COMPUTATIONAL_METRICS) & ~metrics.isin(DETAILED_COMPUTATIONAL)).values
    difference.loc[is_computational, 'value'] = difference.loc[is_computational, 'value'] / difference.loc[is_computational, 'value_b']
    difference.drop(['value_b', 'value_m'], axis=1, inplace=True)
    for c in TO_FLOAT:
        if not c in difference:
            continue
        difference[c] = difference[c].astype(float)
    for c in TO_INT:
        if not c in difference:
            continue
        difference[c] = difference[c].astype(int)
    difference = filter_df(difference, filter)
    difference = difference.set_index(hps).sort_index().reset_index()
    # difference.sort_values(hps, inplace=True)
    return difference


Это хорошая визуализация отдельно по метрикам, но снизу есть версия плотли

to create archives with graphics

In [63]:
# import shutil
# import os

# # Basic usage
# shutil.make_archive('graphs0.05subsample', 'zip', 'GRAPHS')

# # With full path
# # shutil.make_archive('/path/to/archive_name', 'zip', '/path/to/source_folder')

## Main comparison

In [64]:

import plotly.express as px

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Assuming your dataframe is called `df`
# Required columns: 'sketch_method', 'subsample', 'sketch_outputs', and some value column
def create_plot_for_metric(df):
    # Create subplots - one column, multiple rows
    sketch_methods = df['sketch_method'].unique()
    fig = make_subplots(
        rows=len(sketch_methods), 
        cols=1,
        subplot_titles=sketch_methods,
        vertical_spacing=0.08
    )

    # Add a bar plot for each sketch_method
    for i, method in enumerate(sketch_methods, 1):
        method_data = df[df['sketch_method'] == method]
        
        # Get unique combinations of subsample and sketch_outputs
        subsample_levels = method_data['subsample'].unique()
        sketch_outputs = method_data['sketch_outputs'].unique()
        
        # Create bars for each sketch_outputs within each subsample
        for j, output in enumerate(sketch_outputs):
            output_data = method_data[method_data['sketch_outputs'] == output]
            
            # You'll need to aggregate your data appropriately
            # This assumes you have a 'value' column to plot
            fig.add_trace(
                go.Bar(
                    x=output_data['subsample'],
                    y=output_data['value'],  # Replace with your actual value column
                    name=str(output),
                    legendgroup=str(output),
                    showlegend=(i == 1),  # Only show legend for first subplot
                    marker_color=px.colors.qualitative.Set1[j % len(px.colors.qualitative.Set1)]
                ),
                row=i, col=1
            )

    # Update layout
    fig.update_layout(
        height=300 * len(sketch_methods),
        title_text=f"Sketch Methods Analysis | Metric: {df.metric.iloc[0]}",
        barmode='group',  # This groups bars by subsample with different colors for sketch_outputs
        showlegend=True
    )
    return fig

    

def samplings(df, name=None):
    metrics = df.metric.unique()
    for metric in metrics:
        subset = df[df.metric == metric]

        fig = create_plot_for_metric(subset)
        from pathlib import Path
        if name:
            fig.write_image(Path('/home/leostre/Рабочий стол/py-boost/experiments/cifar10')/ f'{metric}_{name}.jpg')
        fig.show()


# samplings(difference)
        

In [65]:
for k, v in processed.items():
    print(k, v.shape)

sigmoid_5.2.1 (10775, 2)
hyperbolic_5.2.1 (10775, 2)
baselines_5.2.1 (4313, 2)
baselines_5.2 (500, 2)


In [50]:
d = compare_prepare('hyperbolic_5.2', 'baselines_5.2', {'dataset': 'cifar10', 'sketch_method': 'topk', 'lr': 0.1, 
                                                     'smoothing_alpha': 0.9, 'stabilization_threshold': 1.0
                                                     })

samplings(d, 'hyperbolic_over_baseline')


KeyError: 'hyperbolic_5.2'

In [67]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd

def create_metric_dropdown_with_column_subplots(df, title="Sketch Methods Analysis"):
    """
    Create dropdown where each metric shows a COLUMN of bar plots (one subplot per sketch_method)
    Properly using make_subplots
    """
    
    sketch_methods = df['sketch_method'].unique()
    metrics = df['metric'].unique()
    sketch_outputs = df['sketch_outputs'].unique()
    colors = px.colors.qualitative.Set1
    
    # Create a master figure with subplots for the first metric
    fig = make_subplots(
        rows=len(sketch_methods),
        cols=1,
        subplot_titles=[f"Sketch Method: {method}" for method in sketch_methods],
        vertical_spacing=0.08,
        shared_xaxes=True
    )
    
    # Add traces for ALL metrics, but only first metric visible initially
    all_traces = []
    
    for metric_idx, current_metric in enumerate(metrics):
        metric_data = df[df['metric'] == current_metric]
        metric_traces = []
        
        for method_idx, method in enumerate(sketch_methods):
            method_data = metric_data[metric_data['sketch_method'] == method]
            aggregated = method_data.groupby(['subsample', 'sketch_outputs'])['value'].mean().reset_index()
            
            for output_idx, output in enumerate(sketch_outputs):
                output_data = aggregated[aggregated['sketch_outputs'] == output]
                
                trace = go.Bar(
                    x=output_data['subsample'],
                    y=output_data['value'],
                    name=output if (metric_idx == 0 and method_idx == 0) else "",
                    legendgroup=output,
                    visible=(metric_idx == 0),
                    marker_color=colors[output_idx % len(colors)],
                    showlegend=(metric_idx == 0 and method_idx == 0),
                    hovertemplate=(
                        f"Metric: {current_metric}<br>"
                        f"Method: {method}<br>"
                        "Subsample: %{x}<br>"
                        f"Output: {output}<br>"
                        "Value: %{y:.3f}<br>"
                        "<extra></extra>"
                    )
                )
                
                # Add trace to the appropriate subplot
                fig.add_trace(trace, row=method_idx + 1, col=1)
                metric_traces.append(trace)
        
        all_traces.append(metric_traces)
    
    # Create dropdown menu
    dropdown_buttons = []
    for metric_idx, current_metric in enumerate(metrics):
        # Calculate visibility array - show only traces for this metric
        visibility = []
        for i, trace in enumerate(fig.data):
            # Trace belongs to current metric if its index modulo total_traces_per_metric equals metric_idx
            traces_per_metric = len(sketch_methods) * len(sketch_outputs)
            trace_metric_idx = i // traces_per_metric
            visibility.append(trace_metric_idx == metric_idx)
        
        dropdown_buttons.append(
            dict(
                label=current_metric,
                method="update",
                args=[
                    {"visible": visibility},
                    {
                        "title": f"{title} - {current_metric}",
                        "yaxis.title": current_metric,
                        "yaxis2.title": current_metric,
                        # Add more yaxes if you have more subplots
                    }
                ]
            )
        )
    
    # Update layout
    fig.update_layout(
        updatemenus=[{
            "buttons": dropdown_buttons,
            "direction": "down",
            "showactive": True,
            "x": 0.1,
            "y": 1.15,
            "xanchor": "left",
            "yanchor": "top"
        }],
        title=f"{title} - {metrics[0]} - {'%' if metrics[0] in COMPUTATIONAL_METRICS else 'p.p.'}",
        barmode='group',
        height=300 * len(sketch_methods),
        showlegend=True
    )
    
    # Update axes
    fig.update_xaxes(title_text="Subsample Level", row=len(sketch_methods), col=1)
    for i in range(1, len(sketch_methods) + 1):
        fig.update_yaxes(title_text=metrics[0], row=i, col=1)
    
    return fig

baseline = processed['baselines_3.1']
modified = processed['hyperbolic_3.1']
difference = (modified - baseline) * 100

index_computational = difference.reset_index().metric.isin(COMPUTATIONAL_METRICS)
difference.iloc[index_computational] = difference.iloc[index_computational.values, :] / baseline.iloc[index_computational.values, :]

difference_at = difference.reset_index()
difference_at = difference_at[
    (difference_at.lr == '0.001') 
    & (difference_at.dataset == 'age_prediction')
    & (difference_at.sketch_method == 'topk')]

fig = create_metric_dropdown_with_column_subplots(d, 'Hyperbolic over Baseline')

KeyError: 'baselines_3.1'

In [68]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_metric_heatmaps(df, dataset=None, sketch_method=None, lr=None, figsize=(15, 10)):
    """
    Plot separate heatmaps for each metric
    Grid: subsample (rows) x sketch_outputs (columns)
    """
    # Filter data if specific parameters provided
    filtered_df = df.copy()
    if dataset:
        filtered_df = filtered_df[filtered_df.dataset == dataset]
    if sketch_method:
        filtered_df = filtered_df[filtered_df.sketch_method == sketch_method]
    if lr:
        filtered_df = filtered_df[filtered_df.lr == lr]
    
    metrics = filtered_df['metric'].unique()
    metrics = [mtr for mtr in metrics if not mtr in ('mean_nodes',)]
    metric =  [mtr for mtr in metrics if mtr not in COMPUTATIONAL_METRICS] + [mtr for mtr in metrics if mtr in COMPUTATIONAL_METRICS]
    n_metrics = len(metrics)
    
    # Calculate grid layout
    n_cols = min(4, n_metrics)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows / 2))
    axes = axes.flatten() if n_metrics > 1 else [axes]
    
    for i, metric in enumerate(metrics):
        
        if i >= len(axes):
            break
            
        ax = axes[i]
        metric_data = filtered_df[filtered_df['metric'] == metric]
        
        # Create pivot table: subsample vs sketch_outputs
        pivot_table = metric_data.pivot_table(
            values='value',
            index='subsample',
            columns='sketch_outputs',
            aggfunc='mean'  # Use mean if multiple values exist
        )
        
        # Sort for better visualization
        pivot_table = pivot_table.sort_index(ascending=False)  # Reverse subsample order
        if metric in COMPUTATIONAL_METRICS:
            pivot_table = pivot_table.round(1)
        
        # Create heatmap
        sns.heatmap(
            pivot_table,
            ax=ax,
            cmap='viridis',
            annot=True,
            fmt='.3f' if metric not in COMPUTATIONAL_METRICS else '.1f',
            cbar_kws={'label': 'Value'},
            linewidths=0.5,
            linecolor='gray'
        )
        
        ax.set_title(f'Metric: {metric}', fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Sketch Outputs', fontsize=12)
        ax.set_ylabel('Subsample', fontsize=12)
        
        # Rotate x labels for better readability
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)
    
    # Hide empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    # Create super title
    title_parts = []
    if dataset: title_parts.append(f"Dataset: {dataset}")
    if sketch_method: title_parts.append(f"Method: {sketch_method}")
    if lr: title_parts.append(f"LR: {lr}")
    super_title = " | ".join(title_parts) if title_parts else "All Configurations"
    
    plt.suptitle(f'Performance Heatmaps\n{super_title}', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()


In [54]:
# d = compare_prepare('baselines_3.1', None, {"lr": '0.005', 'dataset': 'age_prediction', 'sketch_method': 'topk'})
# Usage
plot_metric_heatmaps(processed['baselines_5.2'].reset_index(), dataset='mnist', lr='0.005', sketch_method='topk' )

ZeroDivisionError: integer division or modulo by zero

In [69]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

def create_metric_scatter_long(df, x_metric, y_metric, signature_flag=False):
    """
    Create a scatter plot from long-format data with two specified metrics.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format with 'metric' and 'value' columns
    x_metric : str
        Name of the metric to plot on X-axis
    y_metric : str
        Name of the metric to plot on Y-axis
    signature_flag : bool
        If True, adds straight lines at zero positions on both axes
    
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    
    # Pivot the data to get metrics as columns
    # Identify all columns that are not 'metric' or 'value'
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    
    # Pivot the dataframe
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'  # Assuming one value per combination
    ).reset_index()
    
    # Filter to keep only rows where both metrics exist
    df_plot = df_pivoted.dropna(subset=[x_metric, y_metric])
    
    # Create a unique combination identifier for coloring
    # List all categorical columns (excluding the metrics)
    categorical_cols = id_vars
    
    # Create a combined category for coloring
    df_plot['color_category'] = df_plot[categorical_cols].astype(str).agg(' | '.join, axis=1)
    
    # Create the scatter plot
    fig = px.scatter(
        df_plot,
        x=x_metric,
        y=y_metric,
        color='color_category',
        hover_data=categorical_cols,
        title=f'Scatter Plot: {x_metric} vs {y_metric}',
        labels={x_metric: x_metric, y_metric: y_metric, 'color_category': 'Configuration'}
    )
    
    # Add zero lines if signature_flag is True
    if signature_flag:
        fig.add_vline(x=0, line_width=1.5, line_dash="dash", 
                      line_color="gray", opacity=0.7)
        fig.add_hline(y=0, line_width=1.5, line_dash="dash", 
                      line_color="gray", opacity=0.7)
    
    # Improve layout
    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(
            title=x_metric,
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False
        ),
        yaxis=dict(
            title=y_metric,
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False
        ),
        legend=dict(
            title="Configuration",
            itemsizing='constant',
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02
        ),
        hovermode='closest'
    )
    
    return fig


def create_metric_scatter_long_advanced(df, x_metric, y_metric, signature_flag=False):
    """
    Advanced version for long-format data with more customization.
    """
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Filter to keep only rows with both metrics
    df_plot = df_pivoted.dropna(subset=[x_metric, y_metric])
    
    # Create unique color categories
    categorical_cols = id_vars
    df_plot['color_id'] = df_plot[categorical_cols].astype(str).agg(' | '.join, axis=1)
    unique_configs = df_plot['color_id'].unique()
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each unique configuration
    for config in unique_configs:
        config_data = df_plot[df_plot['color_id'] == config]
        
        # Create hover text with all configuration details
        hover_text = []
        for _, row in config_data.iterrows():
            hover_info_lines = ["<b>Configuration:</b>"]
            for col in categorical_cols:
                hover_info_lines.append(f"{col}: {row[col]}")
            hover_info_lines.extend([
                "",
                f"<b>{x_metric}: {row[x_metric]:.4f}</b>",
                f"<b>{y_metric}: {row[y_metric]:.4f}</b>"
            ])
            hover_text.append('<br>'.join(hover_info_lines))
        
        fig.add_trace(go.Scatter(
            x=config_data[x_metric],
            y=config_data[y_metric],
            mode='markers',
            name=config[:40] + '...' if len(config) > 40 else config,
            text=hover_text,
            hoverinfo='text',
            marker=dict(
                size=10,
                opacity=0.7,
                line=dict(width=1, color='DarkSlateGrey')
            ),
            showlegend=True
        ))
    
    # Add zero lines if requested
    if signature_flag:
        fig.add_shape(
            type="line", x0=0, x1=0, 
            y0=df_plot[y_metric].min(), y1=df_plot[y_metric].max(),
            line=dict(color="gray", width=1.5, dash="dash")
        )
        fig.add_shape(
            type="line", y0=0, y1=0,
            x0=df_plot[x_metric].min(), x1=df_plot[x_metric].max(),
            line=dict(color="gray", width=1.5, dash="dash")
        )
    
    # Update layout
    fig.update_layout(
        title=f'{x_metric} vs {y_metric}',
        xaxis_title=x_metric,
        yaxis_title=y_metric,
        plot_bgcolor='white',
        xaxis=dict(
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False,
            tickformat='.3f'
        ),
        yaxis=dict(
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False,
            tickformat='.3f'
        ),
        legend=dict(
            title="Configuration",
            itemsizing='constant',
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02,
            font=dict(size=9)
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=11,
            font_family="monospace"
        ),
        margin=dict(r=200)  # Extra margin for legend
    )
    
    return fig


def list_available_metrics(df):
    """
    Helper function to list all available metrics in the long-format dataframe.
    """
    if 'metric' in df.columns:
        metrics = df['metric'].unique()
        print("Available metrics:")
        for metric in metrics:
            print(f"  - {metric}")
        return metrics
    else:
        print("No 'metric' column found in dataframe")
        return []


# Example usage:
# Assuming your dataframe 'df' has columns: dataset, sketch_method, sketch_outputs, 
# smoothing_alpha, stabilization_threshold, subsample, lr, metric, value

# List available metrics
# available_metrics = list_available_metrics(df)

# Create scatter plot comparing two metrics
# fig = create_metric_scatter_long(df, x_metric='accuracy', y_metric='loss', signature_flag=True)
# fig.show()

# Or with advanced version
# fig_advanced = create_metric_scatter_long_advanced(df, x_metric='accuracy', y_metric='loss', signature_flag=True)
# fig_advanced.show()

# If you want to filter specific configurations before plotting:
def filter_and_plot(df, x_metric, y_metric, filters=None, signature_flag=False):
    """
    Apply filters to the dataframe before creating the scatter plot.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str
        Metrics to plot
    filters : dict
        Dictionary of column: value filters to apply
    signature_flag : bool
        Add zero lines or not
    
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    df_filtered = df.copy()
    
    if filters:
        for col, val in filters.items():
            if col in df_filtered.columns:
                df_filtered = df_filtered[df_filtered[col] == val]
    
    return create_metric_scatter_long(df_filtered, x_metric, y_metric, signature_flag)


# Example with filtering:
# fig_filtered = filter_and_plot(
#     df, 
#     x_metric='accuracy', 
#     y_metric='loss',
#     filters={'dataset': 'cifar10', 'lr': '0.001'},
#     signature_flag=True
# )
# fig_filtered.show()

In [56]:
d = compare_prepare('hyperbolic_5.2.1', 'baselines_5.2.1', {})

In [108]:
fig = create_metric_scatter_long(d, 'train_time', 'f1', signature_flag=True)
fig.show()

In [109]:
fig = create_metric_scatter_long_advanced(d, 'inference_time', 'f1', signature_flag=True)
fig.show()

In [74]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Output
from IPython.display import display, clear_output

def create_interactive_metric_scatter(df, x_metric, y_metric, signature_flag=False):
    """
    Create an interactive scatter plot with filtering widgets and flexible coloration.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str
        Metrics to plot
    signature_flag : bool
        Add zero lines or not
    """
    
    # Pivot the data
    df = df.copy()
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Identify categorical columns for filtering
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    # Create filters dictionary
    filters = {}
    
    def create_filter_widget(col):
        """Create a filter widget for a column"""
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        return widgets.SelectMultiple(
            options=unique_vals,
            description=col,
            layout=widgets.Layout(width='300px'),
            style={'description_width': 'initial'}
        )
    
    # Create filter widgets
    filter_widgets = {col: create_filter_widget(col) for col in categorical_cols}
    
    # Create color by selector
    color_by_widget = widgets.Dropdown(
        options=['None'] + categorical_cols + ['color_category_auto'],
        value='color_category_auto',
        description='Color by:',
        layout=widgets.Layout(width='300px'),
        style={'description_width': 'initial'}
    )
    
    # Create output widget for the plot
    output_widget = Output()
    
    def update_plot(change=None):
        """Update function called when filters or color selection changes"""
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Determine coloring
            color_by = color_by_widget.value
            
            if color_by == 'None':
                # All points same color
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric}
                )
                fig.update_traces(marker=dict(color='blue', opacity=0.7))
            elif color_by == 'color_category_auto':
                # Color by unique combination of all categorical columns
                filtered_df['color_category'] = filtered_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color='color_category',
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric, 'color_category': 'Configuration'}
                )
            else:
                # Color by specific column
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color=color_by,
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric, color_by: color_by},
                    color_continuous_scale='Viridis' if filtered_df[color_by].dtype in ['float64', 'int64'] else 'Set1'
                )
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout
            fig.update_layout(
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=True,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=True,
                    zeroline=False
                ),
                hovermode='closest',
                height=600
            )
            
            fig.show()
    
    # Create filter UI
    filter_controls = []
    
    # Group filters in rows of 3
    filter_items = list(filter_widgets.items())
    for i in range(0, len(filter_items), 3):
        row = HBox([widget for _, widget in filter_items[i:i+3]])
        filter_controls.append(row)
    
    # Create control panel
    control_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *filter_controls,
        widgets.HTML("<br><b>Styling Options:</b>"),
        HBox([color_by_widget]),
        widgets.HTML("<br><b>Plot Controls:</b>"),
        widgets.Button(description='Reset All Filters', button_style='warning')
    ])
    
    # Reset button functionality
    reset_button = control_panel.children[-1]
    def reset_filters(b):
        for widget in filter_widgets.values():
            widget.value = []
        color_by_widget.value = 'color_category_auto'
    reset_button.on_click(reset_filters)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display the complete interface
    display(VBox([control_panel, output_widget]))
    
    return filter_widgets, color_by_widget


def create_advanced_interactive_scatter(df, x_metric, y_metric, signature_flag=False):
    """
    More advanced version with additional features like:
    - Multiple color modes (categorical, numerical, custom)
    - Size encoding
    - Opacity control
    - Show/hide legend
    - Export data button
    """
    
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Get all columns for selection
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
    all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
    # Create widgets
    filters = {}
    
    # Create filter widgets with multi-select
    filter_widgets = {}
    for col in categorical_cols:
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        filter_widgets[col] = widgets.SelectMultiple(
            options=unique_vals,
            description=col[:15],
            layout=widgets.Layout(width='250px'),
            style={'description_width': 'initial'}
        )
    
    # Color by widget
    color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics
    color_by_widget = widgets.Dropdown(
        options=color_options,
        value='Auto (All columns)',
        description='Color by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Size by widget
    size_by_widget = widgets.Dropdown(
        options=['None'] + numerical_cols + all_metrics,
        value='None',
        description='Size by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Opacity slider
    opacity_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Opacity:',
        layout=widgets.Layout(width='250px')
    )
    
    # Point size slider
    point_size_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=20,
        description='Point size:',
        layout=widgets.Layout(width='250px')
    )
    
    # Legend toggle
    legend_toggle = widgets.Checkbox(
        value=True,
        description='Show legend',
        layout=widgets.Layout(width='150px')
    )
    
    # Grid toggle
    grid_toggle = widgets.Checkbox(
        value=True,
        description='Show grid',
        layout=widgets.Layout(width='150px')
    )
    
    # Color scale for numerical data
    color_scale_widget = widgets.Dropdown(
        options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
        value='Viridis',
        description='Color scale:',
        layout=widgets.Layout(width='250px')
    )
    
    output_widget = Output()
    
    def update_plot(change=None):
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Determine coloring
            color_by = color_by_widget.value
            
            if color_by == 'None':
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    title=f'{x_metric} vs {y_metric}'
                )
                fig.update_traces(
                    marker=dict(
                        color='blue',
                        opacity=opacity_slider.value,
                        size=point_size_slider.value
                    )
                )
            elif color_by == 'Auto (All columns)':
                filtered_df['color_category'] = filtered_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color='color_category',
                    title=f'{x_metric} vs {y_metric}',
                    labels={'color_category': 'Configuration'}
                )
                fig.update_traces(
                    marker=dict(
                        opacity=opacity_slider.value,
                        size=point_size_slider.value
                    )
                )
            else:
                # Check if the color column is numerical or categorical
                is_numerical = filtered_df[color_by].dtype in ['float64', 'int64']
                
                if is_numerical and color_by not in categorical_cols:
                    # Numerical color
                    fig = px.scatter(
                        filtered_df,
                        x=x_metric,
                        y=y_metric,
                        color=color_by,
                        title=f'{x_metric} vs {y_metric}',
                        color_continuous_scale=color_scale_widget.value,
                        labels={color_by: color_by}
                    )
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        )
                    )
                else:
                    # Categorical color
                    fig = px.scatter(
                        filtered_df,
                        x=x_metric,
                        y=y_metric,
                        color=color_by,
                        title=f'{x_metric} vs {y_metric}',
                        color_discrete_sequence=px.colors.qualitative.Set1,
                        labels={color_by: color_by}
                    )
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        )
                    )
            
            # Apply size encoding if selected
            if size_by_widget.value != 'None':
                size_col = size_by_widget.value
                # Normalize sizes to range [5, 20]
                if size_col in filtered_df.columns:
                    size_vals = filtered_df[size_col].fillna(filtered_df[size_col].median())
                    min_size, max_size = size_vals.min(), size_vals.max()
                    if min_size != max_size:
                        normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
                    else:
                        normalized_sizes = [10] * len(filtered_df)
                    
                    fig.update_traces(
                        marker=dict(
                            size=normalized_sizes,
                            sizemode='area',
                            sizeref=2.*max(normalized_sizes)/(40**2)
                        )
                    )
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout based on toggles
            fig.update_layout(
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                hovermode='closest',
                height=600,
                showlegend=legend_toggle.value
            )
            
            # Add hover information
            hover_template = "<b>Configuration</b><br>"
            for col in categorical_cols:
                hover_template += f"{col}: %{{customdata[{categorical_cols.index(col)}]}}<br>"
            hover_template += f"<br><b>{x_metric}</b>: %{{x:.4f}}<br>"
            hover_template += f"<b>{y_metric}</b>: %{{y:.4f}}<br>"
            hover_template += "<extra></extra>"
            
            # Add customdata for hover
            fig.update_traces(
                customdata=filtered_df[categorical_cols],
                hovertemplate=hover_template
            )
            
            fig.show()
    
    # Create UI layout
    # Left panel with filters
    filter_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *[HBox([widget]) for widget in filter_widgets.values()]
    ])
    
    # Right panel with styling options
    style_panel = VBox([
        widgets.HTML("<b>Styling:</b>"),
        color_by_widget,
        size_by_widget,
        color_scale_widget,
        opacity_slider,
        point_size_slider,
        legend_toggle,
        grid_toggle
    ])
    
    # Main control panel
    control_panel = HBox([filter_panel, style_panel])
    
    # Add reset button
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    
    def reset_all(b):
        for widget in filter_widgets.values():
            widget.value = []
        color_by_widget.value = 'Auto (All columns)'
        size_by_widget.value = 'None'
        opacity_slider.value = 0.7
        point_size_slider.value = 8
        legend_toggle.value = True
        grid_toggle.value = True
        color_scale_widget.value = 'Viridis'
    
    reset_button.on_click(reset_all)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    size_by_widget.observe(update_plot, names='value')
    opacity_slider.observe(update_plot, names='value')
    point_size_slider.observe(update_plot, names='value')
    legend_toggle.observe(update_plot, names='value')
    grid_toggle.observe(update_plot, names='value')
    color_scale_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display complete interface
    display(VBox([control_panel, reset_button, output_widget]))
    
    return filter_widgets, color_by_widget, size_by_widget


# Helper function to create a static version with dropdown filters using Plotly's built-in features
def create_static_with_filters(df, x_metric, y_metric, signature_flag=False):
    """
    Create a static Plotly figure with dropdown filters (no ipywidgets required)
    """
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Create color category
    df_pivoted['color_category'] = df_pivoted[id_vars].astype(str).agg(' | '.join, axis=1)
    
    # Create figure
    fig = px.scatter(
        df_pivoted,
        x=x_metric,
        y=y_metric,
        color='color_category',
        title=f'{x_metric} vs {y_metric}',
        labels={'color_category': 'Configuration'}
    )
    
    # Add filters as dropdown menus
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    buttons = []
    for col in categorical_cols:
        unique_vals = ['All'] + sorted(df_pivoted[col].dropna().unique())
        for val in unique_vals:
            buttons.append(
                dict(
                    method='restyle',
                    label=f'{col}: {val}',
                    args=[{'visible': [val == 'All' or df_pivoted[col].iloc[i] == val for i in range(len(df_pivoted))]}]
                )
            )
    
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=buttons[:10],  # Limit to first 10 for performance
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
        ]
    )
    
    # Add zero lines if requested
    if signature_flag:
        fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
        fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
    
    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(gridcolor='lightgray', showgrid=True, zeroline=False),
        yaxis=dict(gridcolor='lightgray', showgrid=True, zeroline=False),
        height=600
    )
    
    return fig


# Example usage:
# For interactive version (requires ipywidgets):
# create_interactive_metric_scatter(df, 'accuracy', 'loss', signature_flag=True)

# For advanced version with more controls:
# create_advanced_interactive_scatter(df, 'accuracy', 'loss', signature_flag=True)

# For static version with dropdowns (no ipywidgets needed):
# fig = create_static_with_filters(df, 'accuracy', 'loss', signature_flag=True)
# fig.show()

In [71]:
d = compare_prepare('sigmoid_5.2.1', 'baselines_5.2.1', {
#     'dataset': 'cifar10', 
                                                       #  'sketch_method': 'topk', 
                                                        # 'lr': 0.1, 
                                                   #   'smoothing_alpha': 0.9, 
                                                    # 'stabilization_threshold': 1.0
                                                     })

In [86]:
sigm = (processed['sigmoid_5.2.1'].reset_index())
flatten_index(sigm)
baseline = processed['baselines_5.2.1'].reset_index()
flatten_index(baseline)

In [87]:
baseline[(baseline.dataset == 'yeast') & (baseline.metric == 'accuracy')]

,dataset,sketch_method,sketch_outputs,subsample,lr,metric,value,std
3363,yeast,topk,1,0.05,0.005,accuracy,0.544479,0.032946
3382,yeast,topk,1,0.05,0.1,accuracy,0.545157,0.030332
3401,yeast,topk,1,0.25,0.005,accuracy,0.619301,0.032352
3420,yeast,topk,1,0.25,0.1,accuracy,0.605810,0.027193
3439,yeast,topk,1,0.5,0.005,accuracy,0.616601,0.026903
3458,yeast,topk,1,0.5,0.1,accuracy,0.613912,0.029257
3477,yeast,topk,1,0.75,0.005,accuracy,0.614576,0.025677
3496,yeast,topk,1,0.75,0.1,accuracy,0.615930,0.033239
3515,yeast,topk,1,1.0,0.005,accuracy,0.605815,0.030838
3534,yeast,topk,1,1.0,0.1,accuracy,0.610536,0.024499


In [84]:
sigm[(sigm.dataset == 'yeast') & (sigm.metric == 'accuracy')]

,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,std
8275,yeast,topk,1,0.9,0.5,0.05,0.005,accuracy,0.573480,0.044059
8300,yeast,topk,1,0.9,0.5,0.05,0.1,accuracy,0.574813,0.024346
8325,yeast,topk,1,0.9,0.5,0.25,0.005,accuracy,0.611209,0.026826
8350,yeast,topk,1,0.9,0.5,0.25,0.1,accuracy,0.618619,0.034683
8375,yeast,topk,1,0.9,0.5,0.5,0.005,accuracy,0.616605,0.032564
...,...,...,...,...,...,...,...,...,...,...
10650,yeast,topk,7,0.9,1.0,0.5,0.1,accuracy,0.597723,0.022863
10675,yeast,topk,7,0.9,1.0,0.75,0.005,accuracy,0.605831,0.036698
10700,yeast,topk,7,0.9,1.0,0.75,0.1,accuracy,0.601097,0.027717
10725,yeast,topk,7,0.9,1.0,1.0,0.005,accuracy,0.606488,0.037208


In [76]:
d.dataset.unique()

array(['age_prediction', 'genbase', 'mediamill', 'rt_iot2022', 'yeast'],
      dtype=object)

In [ ]:
create_interactive_metric_scatter(d, 'train_time', 'f1', signature_flag=True)

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='300px'), options=('age_prediction', 'genbase', 'mediamill', 'rt_iot2022', 'yeast'), style=DescriptionStyle(description_width='initial'), value=()),
  'lr': SelectMultiple(description='lr', layout=Layout(width='300px'), options=(np.float64(0.005), np.float64(0.1)), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='300px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'subsample': SelectMultiple(description='subsample', layout=Layout(width='300px'), options=(np.float64(0.05), np.float64(0.25), np.float64(0.5), np.float64(0.75), np.float64(1.0)), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='300px'), options=(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int6


   1350     )

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['dataset', 'lr', 'sketch_method', 'subsample', 'sketch_outputs', 'smoothing_alpha', 'stabilization_threshold', 'adaptive_threshold_applied', 'adaptive_threshold_fallback', 'exact_match', 'f1_micro', 'mean_leaves', 'mean_nodes', 'ntrees', 'precision', 'recall', 'threshold_is_vector', 'color_category'] but received: train_time


In [91]:
d.metric.unique()

array(['accuracy', 'adaptive_threshold_applied',
       'adaptive_threshold_fallback', 'adaptive_threshold_inference_time',
       'bce_loss', 'exact_match', 'f1', 'f1_macro', 'f1_micro',
       'inference_time', 'mean_leaves', 'mean_nodes',
       'multiclass_logloss', 'ntrees', 'precision', 'recall', 'roc_auc',
       'threshold_is_vector', 'train_time'], dtype=object)

In [92]:
create_interactive_metric_scatter(d, 'inference_time', 'multiclass_logloss', signature_flag=True)

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='300px'), options=('age_prediction', 'genbase', 'mediamill', 'rt_iot2022', 'yeast'), style=DescriptionStyle(description_width='initial'), value=()),
  'lr': SelectMultiple(description='lr', layout=Layout(width='300px'), options=(np.float64(0.005), np.float64(0.1)), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='300px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'subsample': SelectMultiple(description='subsample', layout=Layout(width='300px'), options=(np.float64(0.05), np.float64(0.25), np.float64(0.5), np.float64(0.75), np.float64(1.0)), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='300px'), options=(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int6

In [ ]:
create_interactive_metric_scatter(d, 'inference_time', 'accuracy', signature_flag=True)

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='300px'), options=('age_prediction', 'genbase', 'mediamill', 'rt_iot2022', 'yeast'), style=DescriptionStyle(description_width='initial'), value=()),
  'lr': SelectMultiple(description='lr', layout=Layout(width='300px'), options=(np.float64(0.005), np.float64(0.1)), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='300px'), options=('topk',), style=DescriptionStyle(description_width='initial'), value=()),
  'subsample': SelectMultiple(description='subsample', layout=Layout(width='300px'), options=(np.float64(0.05), np.float64(0.25), np.float64(0.5), np.float64(0.75), np.float64(1.0)), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='300px'), options=(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int6

## Analogues

In [235]:
analogues_names = ['analogues_3.1', 'analogues_lgbm_3.1']

analogues = dict(zip(analogues_names, (get_all_runs_data([analogues_name]) for analogues_name in analogues_names)))

Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: analogues_3.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: analogues_lgbm_3.1


Processing experiments: 100%|██████████| 1/1 [00:00<00:00, 12.12it/s]


In [236]:
def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df

In [237]:
def prepare_analogue(name):
    analogue_df = filter_data(analogues[name])
    ERROR_COLS = [f'fold_{i}_error' for i in range(5)]
    errors = analogue_df[ERROR_COLS].isna().all(axis=1)
    return melt_metrics(analogue_df[errors]).rename(columns={'learning_rate': 'lr'}).groupby(['dataset', 'subsample', 'lr', 'metric']).agg({'value': 'mean'})


In [262]:
processed

{'sigmoid_3.1':                                                                                     value
 dataset        sketch_method sketch_outputs subsample lr    metric                       
 age_prediction proj          1              0.05      0.005 accuracy             0.386650
                                                             f1                   0.372140
                                                             inference_time     129.814991
                                                             mean_leaves         30.269164
                                                             mean_nodes         127.000000
 ...                                                                                   ...
 mnist          topk          7              0.75      0.1   ntrees            1534.000000
                                                             precision            0.987091
                                                             recall        

In [238]:
def prepare_modified(name):
    results = processed[name].reset_index().groupby(['dataset', 'subsample', 'lr', 'metric']).agg({'value': ['max', 'min']})
    results.columns = results.columns.to_flat_index()
    return results 

In [254]:
def comparison_analogues(modified_name, analogue_name, analogue_model='catboost'):
    if not analogue_model:
        analogue_model = analogue_name
    modified_results = prepare_modified(modified_name)
    join_data = prepare_analogue(analogue_name).join(modified_results).reset_index()
    join_data.dropna(axis=0, inplace=True)
    cols = list(join_data.columns)
    cols[-2:]  = ['max', 'min']
    join_data.columns = cols
    join_data[modified_name] = join_data['max']
    join_data = join_data.rename(columns={'value': analogue_model})
    comp_metrics = join_data.metric.isin(COMPUTATIONAL_METRICS)
    join_data.loc[comp_metrics, modified_name] = join_data.loc[comp_metrics, 'min']
    join_data.drop(['max', 'min'], axis=1, inplace=True)
    change_col = f'{modified_name}_over_{analogue_name}'
    join_data[change_col] = (join_data[modified_name] - join_data[analogue_model]) * 100
    join_data[change_col][comp_metrics] = join_data[change_col][comp_metrics] / join_data[analogue_model]
    is_better = ((join_data[change_col] > 0) & (~comp_metrics) ).map({False: '-', True: '+'})
    join_data['is_better'] = is_better
    return join_data


comp = comparison_analogues('sigmoid_3.1', 'analogues_3.1')
comp_lgbm = comparison_analogues('sigmoid_3.1', 'analogues_lgbm_3.1')

In [255]:
comp = comp[~comp.metric.isin(['mean_nodes', 'mean_leaves'])]
comp_lgbm = comp_lgbm[~comp_lgbm.metric.isin(['mean_nodes', 'mean_leaves'])]

In [251]:
COMPARISON_DIR = Path('comparison_analogues')
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

In [252]:
comp.drop('dataset', axis=1).to_csv(COMPARISON_DIR / 'comparison_age_pred_to_catboost_sigmoid.csv')

In [ ]:
prepare_analogue('analogues_3.1').join(results)

value  (value, max)  \
dataset        subsample lr   metric                                     
age_prediction 0.05      0.05 accuracy          0.297500      0.374600   
                              f1                0.213951      0.363259   
                              inference_time   11.021197     18.103744   
                              mean_leaves       0.000000     30.061639   
                              mean_nodes        0.000000    127.000000   
...                                                  ...           ...   
yeast          0.75      0.1  mean_leaves       0.000000           NaN   
                              mean_nodes        0.000000           NaN   
                              precision         0.704321           NaN   
                              recall            0.483847           NaN   
                              train_time      249.887582           NaN   

                                              (value, min)  
dataset        subsample lr   metric                        
age_prediction 0.05      0.05 accuracy            0.356167  
                              f1                  0.351285  
                              inference_time     13.142054  
                              mean_leaves        25.666699  
                              mean_nodes        127.000000  
...                                                    ...  
yeast          0.75      0.1  mean_leaves              NaN  
                              mean_nodes               NaN  
                              precision                NaN  
                              recall                   NaN  
                              train_time               NaN  

[432 rows x 3 columns]

## LaTex formation

In [99]:
total = pd.concat([df.reset_index() for df in
    processed.values()
], axis=0)

In [105]:
humming_loss = total[(total.metric == 'accuracy')].copy() 
humming_loss['value'] = 1 - humming_loss['value']
humming_loss['metric'] = 'humming_loss'

In [108]:
total = pd.concat(
    [total, humming_loss], axis=0
)

In [109]:
total.metric.unique()

array(['accuracy', 'adaptive_threshold_applied',
       'adaptive_threshold_fallback', 'adaptive_threshold_inference_time',
       'bce_loss', 'exact_match', 'f1', 'f1_macro', 'f1_micro',
       'get_indexers_avg_time', 'get_indexers_calls',
       'get_indexers_total_time', 'get_weights_avg_time',
       'get_weights_calls', 'get_weights_total_time', 'inference_time',
       'mean_leaves', 'mean_nodes', 'multiclass_logloss', 'ntrees',
       'precision', 'recall', 'roc_auc', 'threshold_is_vector',
       'train_time', 'humming_loss'], dtype=object)

In [112]:
total.dataset.unique()

array(['age_prediction', 'genbase', 'mediamill', 'rt_iot2022', 'yeast',
       'cifar10'], dtype=object)

In [118]:
METRICS_TO_EXTRACT = [
    'train_time', 
    'inference_time', 
    'f1',
    'f1_macro',
    'f1_micro',
    'humming_loss'
]

DATASETS_TO_EXTRACT = [
    'genbase',
    'yeast', 
    'mnist',
    'age_prediction',
    'mediamill',
    'cifar10'
]

EXPERIMENTAL_ORDER = [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1'
]

In [119]:
TO_MAX = [
    'f1',
    'f1_macro',
    'f1_micro',
]
TO_MIN = [m for m in METRICS_TO_EXTRACT if m not in TO_MAX]

In [127]:

HP_SETUP = {
    'stabilization_threshold': '1.0',
    'lr': "0.1"
}

filtered_total = total[(total.dataset.isin(DATASETS_TO_EXTRACT)) & (total.metric.isin(METRICS_TO_EXTRACT))]

for c, val in HP_SETUP.items():
    filtered_total[c].fillna(val, inplace=True)
    filtered_total = filtered_total[filtered_total[c] == val]

In [130]:
TARGET_DIMENSTIONS = [
    'dataset', 'experiment', 'metric'
]

In [131]:
filtered_total.dtypes

dataset                     object
sketch_method               object
sketch_outputs              object
smoothing_alpha             object
stabilization_threshold     object
subsample                   object
lr                          object
metric                      object
value                      float64
std                        float64
experiment                  object
dtype: object

In [129]:
filtered_total

,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,std,experiment
281,age_prediction,topk,1,0.9,1.0,0.05,0.1,f1,0.346262,0.007658,sigmoid_5.2.1
282,age_prediction,topk,1,0.9,1.0,0.05,0.1,f1_macro,0.345799,0.007664,sigmoid_5.2.1
283,age_prediction,topk,1,0.9,1.0,0.05,0.1,f1_micro,0.355600,0.007674,sigmoid_5.2.1
290,age_prediction,topk,1,0.9,1.0,0.05,0.1,inference_time,22.849715,2.412381,sigmoid_5.2.1
299,age_prediction,topk,1,0.9,1.0,0.05,0.1,train_time,8185.361426,2042.051017,sigmoid_5.2.1
...,...,...,...,...,...,...,...,...,...,...,...
410,cifar10,topk,7,NaN,1.0,0.05,0.1,humming_loss,0.677917,0.007632,baselines_5.2
430,cifar10,topk,7,NaN,1.0,0.25,0.1,humming_loss,0.587083,0.014319,baselines_5.2
450,cifar10,topk,7,NaN,1.0,0.5,0.1,humming_loss,0.551417,0.014821,baselines_5.2
470,cifar10,topk,7,NaN,1.0,0.75,0.1,humming_loss,0.549250,0.009512,baselines_5.2


## Исследование времени операций

In [172]:
hp = processed['sigmoid_exp_smoothing_4.0'].reset_index().dropna()

In [173]:
is_time_metric = hp.metric.map(lambda x: x.endswith('_time'))

In [174]:
hp[(is_time_metric) & (hp.metric != 'inference_time')]

,dataset,sketch_method,sketch_outputs,subsample,lr,metric,value
514,mnist,topk,1,0.05,0.005,get_indexers_avg_time,0.734327
516,mnist,topk,1,0.05,0.005,get_indexers_total_time,11.149722
517,mnist,topk,1,0.05,0.005,get_weights_avg_time,0.276638
519,mnist,topk,1,0.05,0.005,get_weights_total_time,4.202707
527,mnist,topk,1,0.05,0.005,train_time,1214.177747
...,...,...,...,...,...,...,...
1010,mnist,topk,7,0.75,0.1,get_indexers_avg_time,0.787379
1012,mnist,topk,7,0.75,0.1,get_indexers_total_time,180.807680
1013,mnist,topk,7,0.75,0.1,get_weights_avg_time,0.270688
1015,mnist,topk,7,0.75,0.1,get_weights_total_time,62.406023


In [175]:
tt = hp[hp.metric == 'train_time'].groupby(['sketch_outputs', 'subsample', 'lr']).value.aggregate('first')
git = hp[hp.metric == 'get_indexers_total_time'].groupby(['sketch_outputs', 'subsample', 'lr']).value.aggregate('first')
gwt = hp[hp.metric == 'get_weights_total_time'].groupby(['sketch_outputs', 'subsample', 'lr']).value.aggregate('first')

In [176]:
git.name = 'get_indexers'
gwt.name = 'get_weights'

In [177]:
props = pd.concat([
    git / tt * 100,
    gwt / tt * 100
], axis=1)
props.columns = ['get_indexers', 'get_weights']


In [180]:
props.reset_index().groupby(['sketch_outputs', 'subsample',]).agg({'get_indexers': ['max', 'min'], 'get_weights': ['max', 'min']}).round(2)

get_indexers       get_weights      
                                  max   min         max   min
sketch_outputs subsample                                     
1              0.05              1.74  0.92        0.67  0.35
               0.25              2.51  1.58        1.01  0.62
               0.5               2.87  2.08        0.99  0.72
               0.75              2.83  2.38        0.98  0.83
2              0.05              2.03  0.93        0.80  0.39
               0.25              2.49  1.85        1.00  0.73
               0.5               2.87  2.13        0.98  0.71
               0.75              2.85  2.31        0.99  0.79
5              0.05              2.26  0.93        0.92  0.34
               0.25              2.52  2.27        1.01  0.91
               0.5               2.90  2.61        1.00  0.89
               0.75              2.89  2.64        0.99  0.91
7              0.05              2.27  0.94        0.89  0.37
               0.25              2.53  2.26        1.03  0.90
               0.5               2.88  2.57        1.01  0.87
               0.75              2.91  2.53        1.01  0.87

In [ ]:
hp[(is_time_metric) & (hp.metric != 'inference_time')]

In [85]:
import seaborn as sns 
import matplotlib.pyplot as plt 


lgbm_df = df.groupby(['pred_thr', 'prop_keep', 'metric'])['value'].aggregate('mean').reset_index()

# Pivot the data to create a matrix for each metric
pivot_dfs = {}
metrics = df['metric'].unique()

for metric in metrics:
    # Filter for the specific metric
    metric_df = lgbm_df[lgbm_df['metric'] == metric].copy()
    
    # Pivot to create matrix
    pivot = metric_df.pivot_table(
        index='pred_thr', 
        columns='prop_keep', 
        values='value',
        aggfunc='first'
    )
    pivot_dfs[metric] = pivot

# Create heatmaps for each metric
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (metric, pivot_df) in enumerate(pivot_dfs.items()):
    if i < len(axes):
        sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[i],
                   cbar_kws={'label': metric})
        axes[i].set_title(f'{metric} Heatmap')
        axes[i].set_xlabel('prop_keep')
        axes[i].set_ylabel('pred_thr')

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Create separate, more detailed heatmaps for each metric
for metric, pivot_df in pivot_dfs.items():
    plt.figure(figsize=(8, 6))
    
    # Use different colormaps based on metric type
    if metric in ['inference_time', 'train_time']:
        cmap = 'viridis_r'  # Reverse viridis for time (darker = better)
    elif metric in ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']:
        cmap = 'YlOrRd'  # Yellow-Orange-Red for performance metrics
    else:
        cmap = 'coolwarm'
    
    sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap=cmap,
               cbar_kws={'label': metric}, linewidths=1, linecolor='gray')
    plt.title(f'{metric} - pred_thr vs prop_keep', fontsize=14, fontweight='bold')
    plt.xlabel('prop_keep', fontsize=12)
    plt.ylabel('pred_thr', fontsize=12)
    plt.tight_layout()
    plt.show()

# Print summary statistics
print("\n" + "="*50)
print("SUMMARY STATISTICS BY METRIC")
print("="*50)

for metric, pivot_df in pivot_dfs.items():
    print(f"\n{metric.upper()}:")
    print(f"  Best value: {pivot_df.max().max():.6f}")
    print(f"  Worst value: {pivot_df.min().min():.6f}")
    
    if metric in ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']:
        # For performance metrics, find best combination
        best_loc = np.unravel_index(pivot_df.values.argmax(), pivot_df.shape)
        print(f"  Best combination: pred_thr={pivot_df.index[best_loc[0]]}, prop_keep={pivot_df.columns[best_loc[1]]}")
    elif metric in ['inference_time', 'train_time']:
        # For time metrics, lower is better
        best_loc = np.unravel_index(pivot_df.values.argmin(), pivot_df.shape)
        print(f"  Best combination (fastest): pred_thr={pivot_df.index[best_loc[0]]}, prop_keep={pivot_df.columns[best_loc[1]]}")

KeyError: 'pred_thr'